In [1]:
from pathlib import Path
import pandas as pd

folder = Path("results")

dfs = []

for file in folder.glob("*all_experiments_results.parquet"):

    df = pd.read_parquet(file)

    df["dataset"] = file.stem.replace("-all_experiments_results","")

    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [2]:
df["model"] = df["model_id"].str.extract(
    r'ai4i-(.+?)_Layers'
)

df["sequence"] = (
    df["model_id"]
      .str.extract(r'SequenceLength-(\d+)')
      .astype(int)
)

df["iteration"] = (
    df["model_id"]
      .str.extract(r'Iteration-(\d+)')
      .astype(int)
)

In [3]:
metric = "f1"

In [4]:
table = df.pivot_table(

    index=[
        "dataset",
        "sequence",
        "iteration"
    ],

    columns="model",

    values=metric
)

In [5]:
ranks = table.rank(

    axis=1,

    ascending=False,

    method="average"
)

In [6]:
average_rank = ranks.mean()

average_rank = average_rank.sort_values()

print(average_rank)

model
pure-rnn     1.00
pure-gru     2.48
pure-lstm    2.52
hybrid       4.00
dtype: float64


In [7]:
from scipy.stats import friedmanchisquare

stat, p = friedmanchisquare(

    *[table[c] for c in table.columns]

)

print("Friedman")

print(stat)

print(p)

Friedman
135.024
4.4692836274593175e-29


In [10]:
ranks = ranks.reset_index(drop=True)

In [11]:
import scikit_posthocs as sp

nemenyi = sp.posthoc_nemenyi_friedman(ranks)

print(nemenyi)

                 hybrid      pure-gru     pure-lstm      pure-rnn
hybrid     1.000000e+00  2.358186e-08  5.946449e-08  0.000000e+00
pure-gru   2.358186e-08  1.000000e+00  9.986765e-01  5.946449e-08
pure-lstm  5.946449e-08  9.986765e-01  1.000000e+00  2.358186e-08
pure-rnn   0.000000e+00  5.946449e-08  2.358186e-08  1.000000e+00


In [13]:
average_rank.to_csv(
    "outputs/average_rank.csv"
)

nemenyi.to_csv(
    "outputs/nemenyi.csv"
)

In [16]:
from aeon.visualisation import plot_critical_difference

import matplotlib.pyplot as plt

plot_critical_difference(

    average_rank,

    nemenyi

)

plt.savefig(

    "outputs/cd_diagram.png",

    dpi=300,

    bbox_inches="tight"

)

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.